## Similarity function comparison

In [1]:
import os
#virtually move to parent directory
os.chdir("..")

import torch
from sentence_transformers import SentenceTransformer
from sklearn import metrics

import clip
import utils
import data_utils
import similarity

/home/s4yadav/private/workspace/CLIP-dissect/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging


## Settings

In [2]:
similarity_fns = ["cos_similarity", "rank_reorder", "wpmi", "soft_wpmi"]
d_probes = ['cifar100_train', 'broden', 'imagenet_val', 'imagenet_broden']

clip_name = 'ViT-B/16'
target_name = 'resnet50'
target_layer = 'fc'
batch_size = 4
device = 'cuda'
pool_mode = 'avg'
save_dir = 'saved_activations'

In [3]:
# Create a simple wrapper for sentence embeddings using transformers directly
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

class SimpleSentenceTransformer:
    def __init__(self, model_name):
        print(f"Loading {model_name} using transformers...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModel.from_pretrained(model_name, trust_remote_code=True)
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.model.to(self.device)
        self.model.eval()
    
    def encode(self, texts):
        # Convert numpy arrays and other iterables to list of strings
        if isinstance(texts, np.ndarray):
            texts = texts.tolist()
        elif not isinstance(texts, list):
            texts = list(texts)
        if isinstance(texts, str):
            texts = [texts]
        
        # Ensure all elements are strings
        texts = [str(t) for t in texts]
        
        encoded = self.tokenizer(texts, padding=True, truncation=True, return_tensors='pt')
        encoded = {k: v.to(self.device) for k, v in encoded.items()}
        with torch.no_grad():
            output = self.model(**encoded)
            # Use mean pooling
            embeddings = output.last_hidden_state.mean(dim=1)
            embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
        return embeddings.cpu().numpy()

# Load models
try:
    model = SimpleSentenceTransformer('sentence-transformers/all-mpnet-base-v2')
except Exception as e:
    print(f"Failed with all-mpnet-base-v2: {e}")
    print("Using distilbert as fallback...")
    model = SimpleSentenceTransformer('distilbert-base-uncased')

clip_model, _ = clip.load(clip_name, device=device)

with open("data/imagenet_labels.txt", "r") as f:
    cls_id_to_name = f.read().split("\n")

Loading sentence-transformers/all-mpnet-base-v2 using transformers...
Failed with all-mpnet-base-v2: We couldn't connect to 'https://huggingface.co' to load this model, couldn't find it in the cached files and it looks like sentence-transformers/all-mpnet-base-v2 is not the path to a directory containing a config.json file.
Checkout your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
Using distilbert as fallback...
Loading distilbert-base-uncased using transformers...


/home/s4yadav/.conda/envs/clip-rtx/lib/python3.10/site-packages/transformers/modeling_utils.py:367: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(checkpoin

In [4]:
import quantization_utils_awq

# Quantization settings
quantize_enabled = True  # Set to False to disable quantization
quantization_bits = 8    # INT8 quantization
quantization_group_size = 0  # 0 = per-channel, otherwise group-wise

if quantize_enabled:
    target_model_to_use = quantization_utils_awq.quantize_given_mdodel(
        target_name,
        quantization_bits,
        quantization_group_size,
        device
    )
else:
    print(f"Quantization disabled. Using original {target_name} model.")
    target_model_to_use = None  # Will load default in save_activations

Loading resnet50 model for quantization...
Quantizing resnet50 with 8-bit AWQ quantization...
Quantized model saved to saved_activations/resnet50_quantized_8bit.pt
Model quantized successfully!
Model size statistics:
  INT8 parameters: 24.32 MB
  FP32 equivalent: 0.20 MB
  Compression ratio: 0.01x


In [5]:
def save_activations_with_quantization(clip_name, target_name, target_model_to_use, target_layers, d_probe, concept_set, pool_mode, save_dir, batch_size, device):
    # Load models
    clip_model, clip_preprocess = clip.load(clip_name, device=device)

    if quantize_enabled and target_model_to_use is not None:
        # Use quantized model
        target_model = target_model_to_use
        target_preprocess = data_utils.get_target_model(target_name, device)[1]  # Get preprocessor only
    else:
        # Load default model
        target_model, target_preprocess = data_utils.get_target_model(target_name, device)

    # Setup data
    data_c = data_utils.get_data(d_probe, clip_preprocess)
    data_t = data_utils.get_data(d_probe, target_preprocess)

    with open(concept_set, 'r') as f: 
        words = (f.read()).split('\n')

    # Ignore empty lines
    words = [i for i in words if i != ""]

    # Generate text embeddings
    text = clip.tokenize(["{}".format(word) for word in words]).to(device)

    # Get save names
    save_names = utils.get_save_names(
        clip_name=clip_name,
        target_name=target_name,
        target_layer='{}',
        d_probe=d_probe,
        concept_set=concept_set,
        pool_mode=pool_mode,
        save_dir=save_dir
    )

    target_save_name, clip_save_name, text_save_name = save_names

    # Save features using quantized model (if enabled)
    print("Saving CLIP text features...")
    utils.save_clip_text_features(clip_model, text, text_save_name, batch_size)

    print("Saving CLIP image features...")
    utils.save_clip_image_features(clip_model, data_c, clip_save_name, batch_size, device)

    print(f"Saving target activations from {'quantized ' if quantize_enabled else ''}model...")
    utils.save_target_activations(
        target_model,
        data_t,
        target_save_name,
        target_layers,
        batch_size,
        device,
        pool_mode
    )

    print("Activation extraction complete!")


# Cos similarities

In [6]:
concept_set = 'data/3k.txt'

with open(concept_set, 'r') as f:
    words = f.read().split('\n')

for similarity_fn in similarity_fns:
    for d_probe in d_probes:
        # utils.save_activations(clip_name = clip_name, target_name = target_name, target_layers = [target_layer], 
        #                        d_probe = d_probe, concept_set = concept_set, batch_size = batch_size, 
        #                        device = device, pool_mode=pool_mode, save_dir = save_dir)

        save_activations_with_quantization(clip_name = clip_name, 
                       target_name = target_name, 
                       target_model_to_use=target_model_to_use, 
                       target_layers = [target_layer], 
                       d_probe = d_probe, 
                       concept_set = concept_set, 
                       batch_size = batch_size, 
                       device = device, 
                       pool_mode=pool_mode, 
                       save_dir = save_dir
                    )

        save_names = utils.get_save_names(clip_name = clip_name, target_name = target_name,
                                          target_layer = target_layer, d_probe = d_probe,
                                          concept_set = concept_set, pool_mode=pool_mode,
                                          save_dir = save_dir)

        target_save_name, clip_save_name, text_save_name = save_names

        similarities, target_feats = utils.get_similarity_from_activations(target_save_name, clip_save_name, 
                                                                           text_save_name, 
                                                                           eval("similarity.{}".format(similarity_fn)),
                                                                           device=device)

        clip_preds = torch.argmax(similarities, dim=1)
        clip_preds = [words[int(pred)] for pred in clip_preds]

        clip_cos, mpnet_cos = utils.get_cos_similarity(clip_preds, cls_id_to_name, clip_model, model, device, batch_size)
        print("Similarity fn: {}, D_probe: {}".format(similarity_fn, d_probe))
        print("Clip similarity: {:.4f}, mpnet similarity: {:.4f}".format(clip_cos, mpnet_cos))

Files already downloaded and verified
Files already downloaded and verified
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image_features = torch.load(clip_save_name, map_location='cpu').floa

Similarity fn: cos_similarity, D_probe: cifar100_train
Clip similarity: 0.6768, mpnet similarity: 0.5899
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1/1 [00:00<00:00,  4.25it/s]


Similarity fn: cos_similarity, D_probe: broden
Clip similarity: 0.6631, mpnet similarity: 0.5871
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


...

torch.Size([1000, 3000])
Similarity fn: soft_wpmi, D_probe: broden
Clip similarity: 0.7280, mpnet similarity: 0.6201
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:04<00:00, 227.20it/s]


torch.Size([1000, 3000])
Similarity fn: soft_wpmi, D_probe: imagenet_val
Clip similarity: 0.6997, mpnet similarity: 0.3604
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:04<00:00, 208.26it/s]


torch.Size([1000, 3000])
Similarity fn: soft_wpmi, D_probe: imagenet_broden
Clip similarity: 0.7305, mpnet similarity: 0.6219


# Accuracies

In [7]:
def get_topk_acc(sim, k=5):
    correct = 0
    for orig_id in range(1000):
        vals, ids = torch.topk(sim[orig_id], k=k)
        for idx in ids[:k]:
            correct += (int(idx)==orig_id)
    return (correct/1000)*100

def get_correct_rank_mean_median(sim):
    ranks = []
    for orig_id in range(1000):
        vals, ids = torch.sort(sim[orig_id], descending=True)
        
        ranks.append(list(ids).index(orig_id)+1)
        
    mean = sum(ranks)/len(ranks)
    median = sorted(ranks)[500]
    return mean, median

def get_auc(sim):
    max_sim, preds = torch.max(sim.cpu(), dim=1)
    gtruth = torch.arange(0, 1000)
    correct = (preds==gtruth)
    fpr, tpr, thresholds = metrics.roc_curve(correct, max_sim)
    auc = metrics.roc_auc_score(correct, max_sim)
    return auc

In [8]:
concept_set = 'data/imagenet_labels.txt'
with open(concept_set, 'r') as f: 
    words = (f.read()).split('\n')
    

for similarity_fn in similarity_fns:
    for d_probe in d_probes:
        # utils.save_activations(clip_name = clip_name, target_name = target_name, target_layers = [target_layer], 
        #                        d_probe = d_probe, concept_set = concept_set, batch_size = batch_size, 
        #                        device = device, pool_mode=pool_mode, save_dir = save_dir)

        save_activations_with_quantization(clip_name = clip_name, 
                       target_name = target_name, 
                       target_model_to_use=target_model_to_use, 
                       target_layers = [target_layer], 
                       d_probe = d_probe, 
                       concept_set = concept_set, 
                       batch_size = batch_size, 
                       device = device, 
                       pool_mode=pool_mode, 
                       save_dir = save_dir
                    )

        save_names = utils.get_save_names(clip_name = clip_name, target_name = target_name,
                                          target_layer = target_layer, d_probe = d_probe,
                                          concept_set = concept_set, pool_mode=pool_mode,
                  
                                          save_dir = save_dir)

        target_save_name, clip_save_name, text_save_name = save_names

        similarities, target_feats = utils.get_similarity_from_activations(target_save_name, clip_save_name, 
                                                                           text_save_name, 
                                                                           eval("similarity.{}".format(similarity_fn)),
                                                                           device=device)
        
        print("Similarity fn: {}, D_probe: {}".format(similarity_fn, d_probe))
        print("Top 1 acc: {:.2f}%, Top 5 acc: {:.2f}%".format(get_topk_acc(similarities, k=1),
                                                         get_topk_acc(similarities, k=5)))
        
        mean, median = get_correct_rank_mean_median(similarities)
        print("Mean rank of correct class: {:.2f}, Median rank of correct class: {}".format(mean, median))
        print("AUC: {:.4f}".format(get_auc(similarities)))



Files already downloaded and verified
Files already downloaded and verified
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1/1 [00:00<00:00,  4.88it/s]


Similarity fn: cos_similarity, D_probe: cifar100_train
Top 1 acc: 8.70%, Top 5 acc: 25.20%
Mean rank of correct class: 53.88, Median rank of correct class: 21
AUC: 0.5901
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image_features = torch.load(clip_save_name, map_location='cpu').floa

Similarity fn: cos_similarity, D_probe: broden
Top 1 acc: 5.20%, Top 5 acc: 21.00%
Mean rank of correct class: 63.48, Median rank of correct class: 25
AUC: 0.5827


...

Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


100%|██████████| 1000/1000 [00:04<00:00, 212.67it/s]


torch.Size([1000, 1000])
Similarity fn: soft_wpmi, D_probe: imagenet_val
Top 1 acc: 23.60%, Top 5 acc: 46.70%
Mean rank of correct class: 47.06, Median rank of correct class: 7
AUC: 0.7121
Saving CLIP text features...
Saving CLIP image features...
Saving target activations from quantized model...
Activation extraction complete!


/home/s4yadav/private/workspace/CLIP-dissect/utils.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  image_features = torch.load(clip_save_name, map_location='cpu').floa

torch.Size([1000, 1000])
Similarity fn: soft_wpmi, D_probe: imagenet_broden
Top 1 acc: 72.90%, Top 5 acc: 92.00%
Mean rank of correct class: 4.38, Median rank of correct class: 1
AUC: 0.7680
